<a href="https://colab.research.google.com/github/Arashomranpour/finetune/blob/main/finetune_1_noninstruction_domainspecific.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U peft bitsandbytes transformers accelerate trl

In [2]:
!pip install PyMuPDF

In [3]:
# dataset = load_dataset("HuggingFaceFW/fineweb")
# pubmed = load_dataset("ncbi/pubmed")
# dataset = load_dataset("datajuicer/the-pile-pubmed-abstracts-refined-by-data-juicer")
# dataset = load_dataset("open-llm-leaderboard/open_llm_corpus")
# owt = load_dataset("Skylion007/openwebtext")
# ds = load_dataset("armanc/scientific_papers")

NameError: name 'load_dataset' is not defined

In [ ]:
# https://huggingface.co/datasets/Amod/mental_health_counseling_conversations
# https://huggingface.co/datasets/yahma/alpaca-cleaned
# https://huggingface.co/datasets/Open-Orca/OpenOrca
# https://huggingface.co/datasets/tatsu-lab/alpaca
# https://huggingface.co/datasets/OpenAssistant/oasst1

In [ ]:
# https://huggingface.co/datasets/Anthropic/hh-rlhf
# https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned
# https://huggingface.co/datasets/xinlai/Math-Step-DPO-10K

In [4]:
from datasets import Dataset,load_dataset

In [5]:
dataset=load_dataset("roneneldan/TinyStories",split='train')

In [6]:
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 2119719
})


In [7]:
print(dataset[0])

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [7]:
import fitz
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

pdf_texts = extract_text_from_pdf("/content/Metformin.pdf")

pdf_texts

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

In [8]:
import re
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        # Split on double line breaks or long newlines
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:  # ignore too short lines
                paragraphs.append(clean)
    return paragraphs

paragraphs = split_paragraphs(pdf_texts)
data = [{"text": p} for p in paragraphs]

dataset = Dataset.from_list(data)

In [9]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [10]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [11]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [12]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./llama-pharma-domain",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)


In [13]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

not enough source so we go switch to lora

In [14]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [17]:
from transformers import BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize_fn, batched=True)

# Configure 8-bit quantization correctly
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [18]:
# save adapter
non_inst_model_lora = get_peft_model(model, lora_config)

In [19]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [20]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

In [21]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=5, training_loss=9.383116149902344, metrics={'train_runtime': 676.7886, 'train_samples_per_second': 0.03, 'train_steps_per_second': 0.007, 'total_flos': 63629646888960.0, 'train_loss': 9.383116149902344, 'epoch': 5.0})

In [24]:
from peft import PeftModel
import torch

# 1. Determine device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Load the base model first (quantized or not)
base_model_id = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto" if device == "cuda" else None
)

# 3. Load the LoRA adapter
adapter_path = "/content/tinyllama-lora/checkpoint-5"
model = PeftModel.from_pretrained(base_model, adapter_path)
model.to(device)

# 4. Run inference
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Using device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe reduced the incidence of total cholesterol and LDL-C by 18% and 37%, respectively. In a study of 4,925 patients with hypercholesterolemia who received Atorvastatin 10mg/day in combination with Ezetimibe 10mg/day, the incidence of total cholesterol was reduced to 16.3%.
Atrial Fibrill
